# System Design: Part 1
## Scalability & Architecture Fundamentals

**Topics covered:**
1. What is System Design?
2. The System Design Process
3. Horizontal vs Vertical Scaling
4. Load Balancing
5. Sharding

---
> **Goal:** Understand *why* these patterns exist, not just what they are.

## 1. What is System Design?

System design is the process of defining **how components of a system work together** to meet requirements.

When you build a small app, you don't think about this. But the moment your app needs to handle:
- Millions of users
- Terabytes of data
- 24/7 uptime
- Fast responses globally

...the *design* of your system starts mattering as much as the code itself.

### Two core tensions in system design:

| Tension | What it means |
|---|---|
| **Performance vs Cost** | Faster/more reliable = more servers = more money |
| **Simplicity vs Scale** | Simple systems break at scale; complex ones are hard to maintain |

Every system design decision is a **trade-off**. There is no perfect answer — only the right trade-off for your situation.

---
## 2. The System Design Process

When asked to design a system (interview or real-world), follow this structured process:

### Step 1: Clarify Requirements

Before drawing any boxes, ask questions. Requirements fall into two categories:

**Functional requirements** — what the system *does*  
- Users can upload photos
- Users can search by keyword
- System sends notifications

**Non-functional requirements** — how well it *performs*  
- Handle 10,000 requests/second
- 99.9% uptime (about 8.7 hours of downtime/year)
- Respond in under 200ms
- Store 1TB of data

### Step 2: Capacity Estimation (Back-of-envelope)

Rough math to understand the scale you're designing for.

Example — "Design Twitter":
- 100M daily active users
- Each user reads 20 tweets/day → **2 billion reads/day** → ~23,000 reads/second
- Each user writes 1 tweet/day → 100M writes/day → ~1,150 writes/second
- This tells you: **read-heavy system** → optimize for reads

### Step 3: High-Level Design

Draw the big picture first — clients, servers, databases, caches, queues. Don't go deep yet.

```
[Client] → [Load Balancer] → [API Servers] → [Database]
                                          ↘ [Cache]
```

### Step 4: Deep Dive

Pick the hardest/most interesting components and go deep:
- How does the database schema look?
- How does the cache get invalidated?
- What happens when a server goes down?

### Step 5: Identify Bottlenecks & Trade-offs

Every design has weak points. Name them and explain how you'd address them.

> **Key mindset:** A system design interview isn't about finding the "right" answer. It's about showing you can reason through trade-offs.

---
## 3. Horizontal vs Vertical Scaling

When your single server can't handle the load anymore, you have two options:

### Vertical Scaling (Scale Up)

Make the existing machine **bigger and more powerful**.

```
Before:  [Server: 4 CPU, 16GB RAM]
After:   [Server: 32 CPU, 128GB RAM]
```

**Pros:**
- Simple — no code changes needed
- No distributed systems complexity

**Cons:**
- **Hard ceiling** — there's a maximum size machine you can buy
- **Single point of failure** — if that one server dies, everything dies
- Expensive at high end

---

### Horizontal Scaling (Scale Out)

Add **more machines** and distribute the work.

```
Before:  [Server 1]

After:   [Server 1]
         [Server 2]
         [Server 3]
```

**Pros:**
- **Theoretically unlimited** — just keep adding machines
- **Fault tolerant** — if one server dies, others keep running
- Cheaper (commodity hardware vs expensive specialized hardware)

**Cons:**
- More complex — your app must be **stateless** (more on this below)
- Need a load balancer
- Data consistency becomes harder

---

### Stateless vs Stateful Servers

For horizontal scaling to work, your servers must be **stateless** — they don't remember anything about a user between requests.

**Problem with stateful servers:**
```
User logs in → Server 1 saves their session
Next request → Load balancer sends to Server 2
Server 2: "Who are you? I don't know you!"
```

**Solution: Move state out of the server**
```
User logs in → Session saved in Redis (shared cache)
Next request → Any server can look up the session in Redis
```

This is why Redis is so common — it's the shared "memory" for a cluster of stateless servers.

---

### When to use which?

| Situation | Use |
|---|---|
| Small app, quick fix, predictable load | Vertical |
| Production system, unpredictable traffic, need fault tolerance | Horizontal |
| Databases (often) | Vertical first, then specialized horizontal (replication/sharding) |

---
## 4. Load Balancing

A **load balancer** sits in front of your servers and distributes incoming requests across them.

```
                    ┌─────────────────┐
                    │  Load Balancer  │
                    └────────┬────────┘
               ┌─────────────┼─────────────┐
               ▼             ▼             ▼
          [Server 1]    [Server 2]    [Server 3]
```

Without a load balancer, you can't use horizontal scaling — you'd have no way to distribute traffic.

### Load Balancing Algorithms

**Round Robin** — Send each request to the next server in turn (1→2→3→1→2→3...)
- Simple, works well when all servers have similar capacity

**Least Connections** — Send to the server with the fewest active connections
- Better when requests vary in processing time

**IP Hash** — Always send the same user (by their IP) to the same server
- Useful when you *need* some stickiness (though stateless is better)

**Weighted Round Robin** — Servers with more capacity get more requests
- E.g., Server 1 (8 CPU) gets 2x traffic vs Server 2 (4 CPU)

### Health Checks

Load balancers constantly ping servers: "Are you alive?"  
If a server stops responding → load balancer stops sending it traffic.  
This is how you get **automatic failover** — the system heals itself.

### Layer 4 vs Layer 7 Load Balancing

| Type | Works at | Can see | Example |
|---|---|---|---|
| **L4** | Transport layer (TCP/UDP) | IP + port only | AWS NLB |
| **L7** | Application layer (HTTP) | URL, headers, cookies | AWS ALB, Nginx |

L7 is more powerful — e.g., route `/api/*` to API servers and `/static/*` to CDN servers.

### The Load Balancer as a Single Point of Failure

Wait — if the load balancer fails, everything fails. Doesn't that defeat the purpose?

Yes. Solution: **Active-Passive** load balancer pairs. Two LBs — one active, one on standby. If the active one dies, the passive takes over (via DNS or floating IP).

---
## 5. Sharding

Sharding is **horizontal scaling for databases**.

At some point, even the most powerful single database server can't handle your data. Sharding splits your data across multiple database servers.

### The Core Idea

Instead of one database with 1 billion rows:

```
[DB] — 1 billion users
```

Split into 4 shards, each with 250 million rows:

```
[Shard 0] — users 0-249M
[Shard 1] — users 250M-499M
[Shard 2] — users 500M-749M
[Shard 3] — users 750M-999M
```

### Sharding Strategies

**Range-based sharding** — split by range of a key (user ID, date)
- ✅ Simple, easy to do range queries
- ❌ Can create **hot spots** (everyone signing up today goes to the same shard)

**Hash-based sharding** — `shard = hash(user_id) % num_shards`
- ✅ Even distribution
- ❌ Range queries are hard (data is scattered)
- ❌ Adding shards requires **resharding** (recalculating all keys)

**Directory-based sharding** — a lookup service maps each key to a shard
- ✅ Flexible, can move data between shards easily
- ❌ The lookup service becomes a bottleneck/single point of failure

### The Sharding Key Problem

The hardest part is choosing the right **shard key** (which field to shard on).

Bad shard key → hot spots. Example: sharding tweets by `user_id` when Elon Musk has 100M followers. All reads for Elon's tweets hit the same shard.

### A hot spot is when one shard gets way more traffic than the others, making it the bottleneck while the rest sit idle. 

### What Sharding Breaks

Sharding makes your life harder in several ways:

| Feature | Before sharding | After sharding |
|---|---|---|
| `JOIN` queries | Easy | Hard — data is on different machines |
| Transactions | Easy | Hard — can't do ACID across shards |
| Unique IDs | Auto-increment | Need distributed ID generation (e.g., Snowflake IDs) |
| Adding shards | N/A | Requires resharding (painful) |

> **Real talk:** Sharding is a last resort. Most companies hit vertical scaling limits first, then database replication (read replicas), then sharding. Only shard when you truly have no other option — it adds enormous complexity.

## TCP / UDP / HTTP — Quick Concept Notes

**TCP** — Connection-first protocol. Does a 3-way handshake before sending data, then guarantees every packet arrives in order. Slower but reliable. Used for HTTP, file transfers, databases.

**UDP** — No handshake, no guarantees, just fire and forget. Faster but packets can drop or arrive out of order. Used for video calls, gaming, DNS — where speed > perfection.

**HTTP** — A request-response protocol — client asks, server answers. That's the entire model. A text format that defines *what* client and server say to each other (GET /users, 200 OK, etc.). Runs *on top of* TCP. TCP handles delivery; HTTP handles meaning.

> Mental model: HTTP = what we're saying → TCP = how it gets there → IP = where it goes.

---

## ACID Properties — Quick Concept Note

Guarantees that make a database trustworthy for critical data.

- **Atomicity** — All or nothing. If any step in a transaction fails, the whole thing rolls back. No half-writes.
- **Consistency** — Data must always follow defined rules (constraints, foreign keys). A bad transaction gets rejected, not committed.
- **Isolation** — Concurrent transactions don't see each other's in-progress state. Each behaves as if it ran alone.
- **Durability** — Once committed, data survives crashes and restarts. It's on disk, it's permanent.

> SQL DBs (PostgreSQL, MySQL) give full ACID. Most NoSQL DBs trade some of it (usually isolation/consistency) for speed and scale.

---

## Database Replication

Replication = automatically keeping copies of your data on multiple DB servers.
Every write on the **primary** syncs to one or more **replicas**.

**Why?**
- **High availability** — if primary dies, a replica promotes and takes over (failover)
- **Read scaling** — route reads to replicas, writes stay on primary
- **Disaster recovery** — replica in another region = safety net

**Primary-Replica pattern (most common):**
- All **writes** → primary only
- All **reads** → replicas (or primary)
- Replicas are read-only

**Sync vs Async:**
- *Synchronous* — primary waits for replica to confirm before returning. No data loss, but slower.
- *Asynchronous* — primary returns immediately, replica catches up in background. Tiny lag (ms), but faster. Most systems use this.

**The catch — replication lag:**
With async, replicas are slightly behind. Classic problem: user updates their bio (write → primary), immediately views profile (read → replica) → sees stale data.
Fix: route reads to primary for a short window after a write, or use sync replication for critical paths.

> Key distinction: **Replication = copies of the same data** (for availability). **Sharding = splitting data across nodes** (for scale). They solve different problems.

---
## Quick Demonstration: Consistent Hashing

When you use regular hash-based sharding (`shard = hash(key) % N`), adding or removing a shard means **almost all keys need to be remapped**.

**Consistent hashing** solves this — when you add/remove a shard, only ~1/N of keys need to move.

Here's a simplified version to build intuition:

In [1]:
import hashlib

def simple_hash(key: str, num_shards: int) -> int:
    """Regular modulo hashing"""
    return int(hashlib.md5(key.encode()).hexdigest(), 16) % num_shards

keys = [f"user_{i}" for i in range(20)]

print("=== With 3 shards ===")
assignment_3 = {key: simple_hash(key, 3) for key in keys}
for key, shard in assignment_3.items():
    print(f"  {key} → shard {shard}")

print("\n=== With 4 shards (added 1 shard) ===")
assignment_4 = {key: simple_hash(key, 4) for key in keys}

remapped = sum(1 for k in keys if assignment_3[k] != assignment_4[k])
print(f"\nKeys that moved: {remapped}/{len(keys)} ({remapped/len(keys)*100:.0f}%)")
print("This is the resharding problem — almost everything moves!")

=== With 3 shards ===
  user_0 → shard 1
  user_1 → shard 2
  user_2 → shard 0
  user_3 → shard 0
  user_4 → shard 2
  user_5 → shard 1
  user_6 → shard 0
  user_7 → shard 0
  user_8 → shard 2
  user_9 → shard 0
  user_10 → shard 0
  user_11 → shard 1
  user_12 → shard 0
  user_13 → shard 1
  user_14 → shard 2
  user_15 → shard 1
  user_16 → shard 2
  user_17 → shard 2
  user_18 → shard 2
  user_19 → shard 2

=== With 4 shards (added 1 shard) ===

Keys that moved: 9/20 (45%)
This is the resharding problem — almost everything moves!


---
## Summary

| Concept | One-line summary |
|---|---|
| **System design process** | Requirements → Estimation → High-level → Deep dive → Trade-offs |
| **Vertical scaling** | Bigger machine. Simple but has a ceiling and single point of failure |
| **Horizontal scaling** | More machines. Requires stateless servers + load balancer |
| **Stateless servers** | Store session/state in shared cache (Redis), not on the server |
| **Load balancer** | Distributes traffic, does health checks, enables auto-failover |
| **Sharding** | Horizontal scaling for databases. Last resort — adds huge complexity |
| **Shard key** | What you shard on. Bad choice → hot spots. Hard to change later |

### The Scaling Progression (most teams follow this order)

```
1. Vertical scale (easy)
2. Add caching
3. Add read replicas 
4. Horizontal scale app servers + load balancer
5. CDN for static assets
6. Shard the database (only if you must)
```

Most companies never reach step 6.